# Imports and functions 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from pathlib import Path

In [ ]:
base_path = Path("../data/maps_results/")

In [ ]:
path_poscon_openphenom = Path(
    "./openphenom/388_plates_3_combi/results/maps_jcp2022.csv"
)
path_plates_openphenom = Path("./openphenom/388_plates_3_combi/results/maps_plate.csv")
path_poscon_dinov2_g = Path("./dinov2_g/388_plates_3_combi/results/maps_jcp2022.csv")
path_plates_dinov2_g = Path("./dinov2_g/388_plates_3_combi/results/maps_plate.csv")
path_poscon_dinov2_s = Path("./dinov2_s/388_plates_3_combi/results/maps_jcp2022.csv")
path_plates_dinov2_s = Path("./dinov2_s/388_plates_3_combi/results/maps_plate.csv")
path_poscon_resnet50_mean = Path(
    "./resnet50_mean/388_plates_3_combi/results/maps_jcp2022.csv"
)
path_plates_resnet50_mean = Path(
    "./resnet50_mean/388_plates_3_combi/results/maps_plate.csv"
)
path_poscon_resnet50_median = Path(
    "./resnet50_median/388_plates_3_combi/results/maps_jcp2022.csv"
)
path_plates_resnet50_median = Path(
    "./resnet50_median/388_plates_3_combi/results/maps_plate.csv"
)
path_poscon_chada = Path("./chada/388_plates_3_combi/results/maps_jcp2022.csv")
path_plates_chada = Path("./chada/388_plates_3_combi/results/maps_plate.csv")

In [ ]:
# Theorical random values (1/#labels)

embeddings_random_values = {
    "poscon_dinov2_g": 0.125,
    "plates_dinov2_g": 0.002361,
    "poscon_dinov2_s": 0.125,
    "plates_dinov2_s": 0.002361,
    "poscon_chada": 0.125,
    "plates_chada": 0.002361,
    "poscon_openphenom": 0.125,
    "plates_openphenom": 0.002361,
}

In [ ]:
paths = {
    "poscon_openphenom": path_poscon_openphenom,
    "plates_openphenom": path_plates_openphenom,
    "poscon_dinov2_g": path_poscon_dinov2_g,
    "plates_dinov2_g": path_plates_dinov2_g,
    "poscon_dinov2_s": path_poscon_dinov2_s,
    "plates_dinov2_s": path_plates_dinov2_s,
    "poscon_chada": path_poscon_chada,
    "plates_chada": path_plates_chada,
}

In [ ]:
labels = [
    "JCP2022_085227",
    "JCP2022_037716",
    "JCP2022_025848",
    "JCP2022_046054",
    "JCP2022_035095",
    "JCP2022_064022",
    "JCP2022_050797",
    "JCP2022_012818",
]

In [ ]:
def preprocess(path: Path, label: str = "Mean mAP"):
    df_raw = pd.read_csv(base_path / path)
    df_raw.columns = [
        col.replace("mAP", "")
        .replace("raw_", "")
        .replace("(", "")
        .replace(")", "")
        .replace("raw", "Raw")
        .strip()
        for col in df_raw.columns
    ]
    df = df_raw.drop(columns=["Number of Queries"])
    df = df.set_index("Label").astype(float)
    return df.loc[label]

In [ ]:
def plot_curve_comparison(encoders_dict):
    """
    Plot mAP values for multiple encoders with custom styling.
    Highlights specific points for each encoder (No Normalisation, Random, Best Normalisation)
    with unified colors.

    Args:
        encoders_dict: Dictionary where keys are encoder names and values are lists containing:
                       [poscon_values, batch_effect_values, best_normalisation_index].
    """
    import matplotlib.pyplot as plt
    import matplotlib.lines as mlines

    # High resolution figure for publication (dpi=1500)
    plt.figure(figsize=(18, 8), dpi=300)

    # Define marker styles for each encoder
    marker_styles = ["o", "s", "D", "^", "v", "P", "X"]  # Extend as needed
    # Couleurs adaptées pour une publication
    unified_no_norm_color = "#D55E00"  # Couleur unifiée pour "No Normalisation"
    unified_best_norm_color = "#009E73"  # Couleur unifiée pour "Best Normalisation"
    unified_random_color = "#56B4E9"  # Couleur unifiée pour "Random Values" (bleu vif)

    shape_patches = []  # Pour la légende des formes

    # 1. Tracer tous les points en gris clair (contexte général)
    for i, (encoder, values) in enumerate(encoders_dict.items()):
        mean_values_1, mean_values_2, best_normalisation_index = values
        # Attribution d'un marqueur unique par encoder
        marker = marker_styles[i % len(marker_styles)]
        plt.scatter(
            mean_values_2, mean_values_1, color="grey", alpha=0.5, s=80, marker=marker
        )

    # 2. Mettre en évidence les points importants avec des tailles et contours renforcés
    for i, (encoder, values) in enumerate(encoders_dict.items()):
        mean_values_1, mean_values_2, best_normalisation_index = values
        marker = marker_styles[i % len(marker_styles)]

        # Point "No Normalisation"
        plt.scatter(
            mean_values_2["Embeddings_Raw"],
            mean_values_1["Embeddings_Raw"],
            color=unified_no_norm_color,
            alpha=1.0,
            s=180,
            edgecolor="black",
            linewidth=2,
            marker=marker,
            zorder=3,
        )
        # Point "Random Values"
        plt.scatter(
            mean_values_2["Embeddings Random"],
            mean_values_1["Embeddings Random"],
            color=unified_random_color,
            alpha=1.0,
            s=180,
            edgecolor="black",
            linewidth=2,
            marker=marker,
            zorder=3,
        )
        # Point "Best Normalisation"
        plt.scatter(
            mean_values_2[best_normalisation_index],
            mean_values_1[best_normalisation_index],
            color=unified_best_norm_color,
            alpha=1.0,
            s=180,
            edgecolor="black",
            linewidth=2,
            marker=marker,
            zorder=3,
        )

        # Ajouter le marqueur dans la légende s'il n'est pas déjà présent
        if marker not in [line.get_marker() for line in shape_patches]:
            shape_patches.append(
                mlines.Line2D(
                    [],
                    [],
                    color="black",
                    marker=marker,
                    linestyle="None",
                    markersize=10,
                    label=f"{encoder}",
                )
            )

    plt.grid()
    # Ajout des labels
    plt.xlabel("mAP - Positive Control Molecules Retrieval", fontsize=16, labelpad=10)
    plt.ylabel("mAP - Plates Retrieval", fontsize=16, labelpad=10)

    # Augmenter la police des valeurs sur les axes
    plt.xticks(fontsize=16)
    plt.yticks(fontsize=16)

    # Ajout de la légende pour les formes (Source Laboratories)
    plt.legend(
        handles=shape_patches, loc="upper left", bbox_to_anchor=(1.05, 0.5), fontsize=14
    )

    # Optimisation de la mise en page
    plt.tight_layout()
    plt.show()

# Plot

In [ ]:
processed_data = {}

for key, path in paths.items():
    df = preprocess(path)
    if key in embeddings_random_values:
        df["Embeddings Random"] = embeddings_random_values[key]
    processed_data[key] = df

In [ ]:
encoder = "dinov2_s"

df = pd.concat(
    [processed_data[f"plates_{encoder}"], processed_data[f"poscon_{encoder}"]], axis=1
)
df.columns = ["Batch_effect", "Poscon"]
df.sort_values(by="Poscon", ascending=False).head(10)

In [ ]:
encoder = "openphenom"

df = pd.concat(
    [processed_data[f"plates_{encoder}"], processed_data[f"poscon_{encoder}"]], axis=1
)
df.columns = ["Batch_effect", "Poscon"]
df.sort_values(by="Poscon", ascending=False).head(10)

In [ ]:
encoder = "dinov2_g"

df = pd.concat(
    [processed_data[f"plates_{encoder}"], processed_data[f"poscon_{encoder}"]], axis=1
)
df.columns = ["Batch_effect", "Poscon"]
df.sort_values(by="Poscon", ascending=False).head(10)

In [ ]:
encoder = "chada"

df = pd.concat(
    [processed_data[f"plates_{encoder}"], processed_data[f"poscon_{encoder}"]], axis=1
)
df.columns = ["Batch_effect", "Poscon"]
df.sort_values(by="Poscon", ascending=False).head(10)

In [ ]:
all_encoder = {
    "ChAda": [
        processed_data["plates_chada"],
        processed_data["poscon_chada"],
        "Embeddings_Raw__ZCA_N_C__Int",
    ],
    "Dinov2_g": [
        processed_data["plates_dinov2_g"],
        processed_data["poscon_dinov2_g"],
        "Embeddings_Raw__ZCA_C__Int",
    ],
    "Dinov2_s": [
        processed_data["plates_dinov2_s"],
        processed_data["poscon_dinov2_s"],
        "Embeddings_Raw__ZCA_C__Int",
    ],
    "Open_Phenom": [
        processed_data["plates_openphenom"],
        processed_data["poscon_openphenom"],
        "Embeddings_Raw__ZCA_N_C__Int",
    ],

}

In [ ]:
plot_curve_comparison(all_encoder)